# Characterize Circuit Distance & Problematic Errors

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

Why can a code with distance $d=2$ suffer from circuit distance $d_{\text{circuit}} = 1$, and how does `GadgetProfile` pinpoint the exact problematic errors?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Compute the static circuit distance of every gadget in a protocol using `ec.GadgetProfile(gadget).distance()`.
- Inspect `distance().witness` and `effects_of(...)` to trace how a single mid-circuit ancilla fault spreads into an undetected weight-2 logical error.
- Identify which gadgets in `bare-css/v1` suffer from problematic errors (`syndrome`, `prepare_z_all`, `prepare_x_all`).

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Code Distance vs. Circuit Distance

In Module 1, we proved that the $C_4$ code has **code distance $d = 2$**: no single-qubit Pauli error on the 4 data qubits can silently corrupt logical information.
In Module 4, `ec.audit(protocol)` confirmed that every gadget in our baseline `bare-css/v1` protocol implements its ideal logical channel in the **noiseless** case.

Now comes the central question of fault-tolerant circuit design:
> **What is the minimum number of elementary circuit faults inside each gadget required to produce an undetected logical error (or an unflagged weight-$\ge 2$ data error on the output block)?**

We measure this with **`ec.GadgetProfile(gadget).distance()`**. Let's compute the circuit distance of all 6 gadgets in our baseline `C4` protocol!

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Compute Circuit Distance Across All 6 Baseline Gadgets
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

print(f"{'Gadget Name':<18} | {'Circuit Distance':<16} | {'Minimum-Weight Fault Witness'}")
print("-" * 82)
for name, gadget in protocol.layers[0].gadgets.items():
    prof = ec.GadgetProfile(gadget)
    dist = prof.distance()
    print(f"{name:<18} | d_circuit = {dist.value:<6} | {dist.witness}")

Gadget Name        | Circuit Distance | Minimum-Weight Fault Witness
----------------------------------------------------------------------------------
cx_all             | d_circuit = 2      | X_4 after call 0; X_5 after call 1
measure_x_all      | d_circuit = 2      | X_0 after call 0; X_1 after call 1
measure_z_all      | d_circuit = 2      | flip call 0 readout 0; flip call 1 readout 0
prepare_x_all      | d_circuit = 1      | X_5 after call 18


prepare_z_all      | d_circuit = 1      | X_4 after call 7
syndrome           | d_circuit = 1      | X_4 after call 3


## Anatomy of a Problematic Error in `syndrome`

Look at the table above:
- `cx_all`, `measure_z_all`, and `measure_x_all` are transversal, so a single fault affects at most one physical qubit per block ($d_{\text{circuit}} = 2$).
- **`syndrome`, `prepare_z_all`, and `prepare_x_all` have $d_{\text{circuit}} = 1$!**

Let's zoom in on `syndrome` using `syndrome_profile.distance()` and `syndrome_profile.effects_of(...)`.
In the baseline `syndrome` circuit:
```
call[00]: R 4          call[02]: H 4          call[07]: CX 4 2
call[01]: R 5          call[03]: CX 4 0       call[08]: CX 0 5
                       call[04]: CX 4 1       call[09]: CX 4 3
                       call[05]: CX 2 5       call[10]: CX 1 5
                       call[06]: CX 3 5       call[11]: H 4
```
What happens if a single **`X_4`** fault occurs right after `call[4]` (`CX 4 1`), or if we inspect `syndrome_distance.witness`?
Let's run `syndrome_profile.effects_of(...)` to see its exact effect on checks, readouts, and output logical frames!

In [3]:
# Trace the Problematic Error in Baseline `syndrome` with `effects_of`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")

syndrome_profile = ec.GadgetProfile(protocol.layers[0].gadgets["syndrome"])
syndrome_distance = syndrome_profile.distance()
calls = protocol.layers[0].gadgets["syndrome"].circuit.calls()

witness_fault = syndrome_distance.witness.product
witness_effect = syndrome_profile.effects_of([witness_fault])[0]
print("Single-fault witness returned by `syndrome_profile.distance()`:")
print("  witness :", syndrome_distance.witness)
print("  effect  :", witness_effect)

# Inspect all distance-1 fault witnesses in baseline `syndrome`:
all_faults = [w.product for w in syndrome_distance.witnesses]
all_effects = syndrome_profile.effects_of(all_faults)
print(f"\nAll {len(all_faults)} distance-1 fault witnesses in baseline `syndrome`:")
for fault, eff in zip(all_faults, all_effects):
    loc = fault.locations[0]
    call = calls[loc.after_call]
    print(f"  {fault} (after call[{loc.after_call}] `{call.mnemonic} {call.operands}`) -> checks={eff.checks}, frames={eff.frames}")

Single-fault witness returned by `syndrome_profile.distance()`:
  witness : X_4 after call 3
  effect  : ['out[0].z[0]']

All 4 distance-1 fault witnesses in baseline `syndrome`:
  X_4 after call 3 (after call[3] `CX [4, 1]`) -> checks=(), frames=(Reference('out[0].z[0]'),)
  X_2 X_4 after call 4 (after call[4] `CX [4, 2]`) -> checks=(), frames=(Reference('out[0].z[0]'),)
  X_5 after call 10 (after call[10] `CZ [5, 1]`) -> checks=(), frames=(Reference('out[0].x[1]'),)
  Z_2 X_5 after call 11 (after call[11] `CZ [5, 2]`) -> checks=(), frames=(Reference('out[0].x[1]'),)


Look closely at the output of `syndrome_profile.effects_of(...)`:
- **`checks: []`**: Neither stabilizer check fires! Why? Because the fault occurred on ancilla `4` ($X$-type) or ancilla `5` ($Z$-type) and propagated through the remaining CNOTs onto **two** data qubits (`X_2 X_3` or `Z_0 Z_1`), which is a valid logical operator of $C_4$ that commutes with all stabilizers!
- **`frames: [outputs[0]:Z_0]` (or `outputs[0]:X_1`)**: The output logical state is silently flipped!

This mechanism—where a single fault on an ancilla qubit during multi-qubit parity measurement spreads through subsequent CNOT gates into a higher-weight data error—is called a **problematic error**.

## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why does a single $X_4$ fault midway through measuring the weight-4 stabilizer $X_0 X_1 X_2 X_3$ cause an undetectable logical error in $C_4$?</summary>

<div style="padding:0.1em 1em;">

Ancilla 4 is prepared in $|+\rangle$ and acts as the control of four CNOTs (`CX 4 0`, `CX 4 1`, `CX 4 2`, `CX 4 3`). An $X_4$ fault after the first two CNOTs propagates through the last two CNOTs onto data qubits 2 and 3, creating the weight-2 data error $X_2 X_3$. Because $X_2 X_3 = \bar{X}_0 \cdot (X_0 X_1 X_2 X_3)$ is a weight-2 logical operator of $C_4$, it commutes with all stabilizers and is completely invisible to future syndrome checks!

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How does `GadgetProfile.effects_of(fault)` help distinguish a harmless detectable fault from a fatal distance-reducing fault?</summary>

<div style="padding:0.1em 1em;">

`effects_of(fault)` returns a `FaultEffect` with `.checks`, `.readouts`, and `.frames`. If `.frames` (or `.readouts`) is non-empty while `.checks` (and any rejection flag in `.readouts`) is empty, the fault silently corrupts logical information without triggering any parity check or flag.

</div>

</details></div>

## Exercise: Compute the Baseline Syndrome Circuit Distance

Use `ec.GadgetProfile(syn).distance()` to compute and return the static circuit distance of the unflagged `bare-css/v1` `syndrome` gadget on $C_4$.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def baseline_syndrome_circuit_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    # Return the integer circuit distance of syn
    return ...

**Hint**

Call `ec.GadgetProfile(syn).distance().value`.

In [ ]:
from qdk import ec
from _unit import exercise, simulation


@exercise
def baseline_syndrome_circuit_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    return ec.GadgetProfile(syn).distance().value

In the naive `bare-css/v1` schedule, a single ancilla fault halfway through the 4 CNOT gates propagates onto 2 data qubits without flipping any syndrome readout, reducing the circuit distance to `1`.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 5
- **Learning Objectives:**
  1. Compute `ec.GadgetProfile(g).distance()` across all 6 gadgets in the baseline `bare-css/v1` protocol.
  2. Identify why `cx_all`, `measure_z_all`, and `measure_x_all` have circuit distance 2, while `syndrome`, `prepare_z_all`, and `prepare_x_all` drop to **circuit distance 1**.
  3. Inspect `syndrome_distance.witness` and `syndrome_profile.effects_of(...)` to trace how a single ancilla fault (`X_4` after call 4, i.e., midway through the X-ancilla CNOTs) propagates onto two data qubits (`X_2 X_3`), flipping logical $Z$ on the output block (`outputs[0]:Z_0`) with **zero** triggered checks (`checks: []`)!
- **Pedagogical Highlight:** This is the "Aha!" moment of the workshop. Walk through the 4 CNOTs from ancilla 4 (`CX 4 0`, `CX 4 1`, `CX 4 2`, `CX 4 3`): an $X_4$ fault midway through spreads $X$ onto two data qubits, which is already a weight-2 logical operator of $C_4$!

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../04-validate-and-audit/validate_and_audit.workbook.ipynb" title="Previous unit: Validate & Fill Gadgets with Audit and GadgetProfile" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Validate & Fill Gadgets with Audit and GadgetProfile</span></span></a><a href="../06-fault-tolerant-gadgets/fault_tolerant_gadgets.workbook.ipynb" title="Next unit: Upgrade Gadgets for Fault Tolerance" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Upgrade Gadgets for Fault Tolerance</span></span><span aria-hidden="true">&#8250;</span></a></div>